In [ ]:
# Parameters
import os, json
output_dir = os.environ.get("ORION_SIGNALS_DIR", "../signals")
dry_run = False
os.makedirs(output_dir, exist_ok=True)


In [ ]:
# -*- coding: utf-8 -*-
# -- QUARTERLY: tickers whose 09:30->09:45 move repeats on the same calendar day every 3 months ----
#
# THE QUESTION (the operator, 2026-11-17): "числа завжди однакові +-2 дні, місяць міняється з
# кроком 3 ... якщо вибірка на дистанції показує 70% успішності - це наш кейс. Особливо цікавлять
# найсвіжіші календарні дати. Якщо останні 3 в одну сторону - це дуже добре."
#
# So an ANCHOR is a pair (day-of-month D, month-phase P), where P = month % 3. The months a phase
# visits are exactly three apart: P=2 is Feb, May, Aug, Nov - the operator's own 16.08 / 16.11 /
# 16.02 example, which is day 16, phase 2. Every month of that phase in the data is one EXPECTED
# occurrence; the session that actually carries it may land up to `day_tol` days either side,
# because the date falls on a weekend often enough that a strict match would miss half of them.
#
# WHAT IS MEASURED. The move is DEV at 10:00 minus DEV at 09:30 - the ticker's deviation from its
# own benchmark, so the benchmark's own move is already removed. That is not a refinement, it is
# what makes the question answerable: measured on this file, the largest RAW 09:30->09:45 moves of
# META, AAPL and NVDA are the same handful of dates (2024-08-05, the April-2025 tariff days,
# 2026-07-02) because those are market-wide. A search on raw moves returns those macro days for
# every ticker and they then look "quarterly" by coincidence.
#
# THE FIELD IS `deviation` - the operator's own choice - which is DEV in PERCENTAGE POINTS:
# Stack% - beta_day x Bench%, rounded to 2 dp in the file. `dev_sig` is the identical quantity
# already divided by sigma_day; either can be passed to load_quarterly_bars.
#
# Because pp are not comparable between tickers (a volatile name moves 2 pp most mornings, a quiet
# one never does), the search STANDARDISES the move per ticker before applying the floor - see
# find_quarterly's own `standardise`. Without that the 70% rule would rank tickers by volatility
# instead of by pattern. mean_move is still reported in pp, which is the readable unit.
#
# WHY THERE IS A SECOND STAGE. There are 31 x 3 = 93 anchors per ticker and thousands of tickers,
# while 2.72 years of history gives at most 11 occurrences per anchor. P(>=7 of 9 | coin flip) is
# 0.09, so a plain "70% hit rate" filter over half a million (ticker, anchor) pairs returns tens of
# thousands of patterns made of nothing. `quarterly_pvalues` re-runs the WHOLE anchor search on the
# same ticker's own moves, shuffled across its own dates, and reports how often chance does as
# well. Only the first stage is cheap enough to run on everything, so the null runs on survivors.
from __future__ import annotations

import numpy as np
import pandas as pd

QUARTER_STEP_MONTHS = 3
PHASES = (0, 1, 2)


def _robust_scale(x):
    """1.4826 * MAD - equal to sigma for normal data, but not inflated by the very days the search
    is looking for. Falls back to the plain std when the median absolute deviation is zero, which
    happens on a quiet ticker because `deviation` is rounded to 2 dp in the file."""
    med = np.median(x)
    mad = float(np.median(np.abs(x - med))) * 1.4826
    if np.isfinite(mad) and mad > 0:
        return mad
    sd = float(np.std(x))
    return sd if (np.isfinite(sd) and sd > 0) else 0.0


# =================================================================================================
# 1) the bars: one DEV move per ticker-session
# =================================================================================================
def load_quarterly_bars(final_path, *, dev_field="deviation", px_field="Stack%",
                        open_min=9 * 60 + 30, late_min=9 * 60 + 45, tol=3, log_every=60):
    """-> DataFrame[ticker, day, move, move_px].

    `move` is the dev_field's 09:30->09:45 change (the ticker's own move, benchmark removed) and
    `move_px` the px_field's (the raw price move). Both are carried because the search wants an
    event to be visible on both axes - see find_quarterly's `require_both`.

    Only ~1.5% of the file's minutes are wanted, so the window mask runs in Arrow and only the
    surviving rows are ever handed to pandas.

    `tol` is a DATA tolerance, not part of the question: a ticker with no print exactly at 09:30
    uses the nearest print within +/-tol minutes, and a ticker with none at all that session simply
    has no row. It is NOT the +/-2 CALENDAR days - that lives in the anchor search below.
    """
    import pyarrow as pa, pyarrow.parquet as pq, pyarrow.compute as pc
    import time

    pf = pq.ParquetFile(final_path)
    for f in (dev_field, px_field):
        if f not in pf.schema.names:
            raise KeyError("final.parquet has no '%s' column" % f)

    parts, t0, rows_in = [], time.time(), 0
    for ci in range(pf.num_row_groups):
        tb = pf.read_row_group(ci, columns=["ticker", "dt", dev_field, px_field])
        rows_in += tb.num_rows
        dts = tb.column("dt")
        hh = pc.cast(pc.utf8_slice_codeunits(dts, 11, 13), pa.int32())
        mm = pc.cast(pc.utf8_slice_codeunits(dts, 14, 16), pa.int32())
        t = pc.add(pc.multiply(hh, 60), mm)
        near = pc.or_(pc.and_(pc.greater_equal(t, open_min - tol), pc.less_equal(t, open_min + tol)),
                      pc.and_(pc.greater_equal(t, late_min - tol), pc.less_equal(t, late_min + tol)))
        keep = pc.and_(pc.fill_null(near, False), pc.is_valid(tb.column(dev_field)))
        small = tb.filter(keep)
        if small.num_rows == 0:
            continue
        df = small.to_pandas()
        df["smin"] = pc.filter(t, keep).to_numpy(zero_copy_only=False)
        df["day"] = df["dt"].str.slice(0, 10)
        parts.append(df.drop(columns=["dt"]))
        if log_every and (ci + 1) % log_every == 0:
            print("[rg %4d/%d] read=%s kept=%s %.0fs"
                  % (ci + 1, pf.num_row_groups, format(rows_in, ","),
                     format(sum(len(p) for p in parts), ","), time.time() - t0))
    if not parts:
        return pd.DataFrame(columns=["ticker", "day", "dev_open", "dev_late", "move"])

    d = pd.concat(parts, ignore_index=True)
    for f in (dev_field, px_field):
        d[f] = pd.to_numeric(d[f], errors="coerce")

    def pick(target):
        w = d[(d["smin"] >= target - tol) & (d["smin"] <= target + tol)].copy()
        w["dd"] = (w["smin"] - target).abs()
        w = w.sort_values(["ticker", "day", "dd"], kind="stable")
        w = w.drop_duplicates(["ticker", "day"], keep="first").set_index(["ticker", "day"])
        return w[dev_field], w[px_field]

    (o_dev, o_px), (c_dev, c_px) = pick(open_min), pick(late_min)
    out = pd.DataFrame({"o_dev": o_dev, "c_dev": c_dev, "o_px": o_px, "c_px": c_px}).dropna()
    out["move"] = out["c_dev"] - out["o_dev"]
    out["move_px"] = out["c_px"] - out["o_px"]
    return out.reset_index()[["ticker", "day", "move", "move_px"]]


# =================================================================================================
# 2) the anchor machinery
# =================================================================================================
def _anchor_slots(dates, day_tol):
    """-> {(D, P): [array of row indices, one array per quarter-month present]}.

    One SLOT is one expected occurrence: the sessions that fall within `day_tol` calendar days of
    day D in a month of phase P. A slot can hold several sessions (D on a Sunday pulls in Fri and
    Mon both) and the scorer picks one of them; a month with no session in the window yields no
    slot at all, so a ticker that simply was not trading then is not punished for a miss.
    """
    y = dates.year.to_numpy()
    m = dates.month.to_numpy()
    ordv = dates.view("int64") // 86_400_000_000_000      # calendar days since epoch, for ±tol
    months = sorted(set(zip(y.tolist(), m.tolist())))

    slots = {}
    for D in range(1, 32):
        for P in PHASES:
            per_month = []
            for (yy, mm) in months:
                if mm % QUARTER_STEP_MONTHS != P:
                    continue
                try:
                    centre = pd.Timestamp(year=yy, month=mm, day=D)
                except ValueError:
                    continue                                # e.g. 31 Feb - that month has no anchor
                c = centre.value // 86_400_000_000_000
                idx = np.flatnonzero(np.abs(ordv - c) <= day_tol)
                if idx.size:
                    per_month.append((c, idx))
            if len(per_month) >= 2:
                slots[(D, P)] = per_month
    return slots


def _score_anchor(z, per_month, min_abs_z, raw=None, anchor_day=None, days=None,
                  z2=None, raw2=None, min_abs_z2=None):
    """-> dict for one (ticker, anchor), or None when nothing fires.

    THE PICK RULE. Among the sessions in a slot, the occurrence is the one that FIRES and sits
    closest to the anchor day; if none fires, the closest session is taken and counted as a miss.
    Picking the largest |move| instead - as an earlier version did - let a stray neighbour outrank
    the anchor day on a quiet quarter, which scattered the day-of-month series of events that were
    in fact perfectly regular. The null picks the same way, so the rule cannot flatter the score.
    """
    def _fires(ii):
        """An occurrence has to show on BOTH axes, in the same direction, when z2 is given."""
        f = np.abs(z[ii]) >= min_abs_z
        if z2 is not None:
            f &= (np.abs(z2[ii]) >= min_abs_z2) & (np.sign(z2[ii]) == np.sign(z[ii]))
        return f

    picks = []
    for c, idx in per_month:
        fires = idx[_fires(idx)]
        pool = fires if fires.size else idx
        if days is not None and anchor_day is not None:
            j = pool[np.argmin(np.abs(days[pool] - anchor_day))]
        else:
            j = pool[np.argmax(np.abs(z[pool]))]
        picks.append((c, j, z[j]))
    n = len(picks)
    if n == 0:
        return None

    order = np.argsort([p[0] for p in picks])              # oldest -> newest
    picks = [picks[i] for i in order]
    vals = np.array([p[2] for p in picks], dtype="float64")
    raw_o = np.array([raw[p[1]] for p in picks], dtype="float64") if raw is not None else vals

    sel_idx = np.array([p[1] for p in picks])
    fired = _fires(sel_idx)
    if not fired.any():
        return None
    vals2 = np.array([z2[i] for i in sel_idx], dtype="float64") if z2 is not None else None
    raw2_o = np.array([raw2[i] for i in sel_idx], dtype="float64") if raw2 is not None else None

    # the side the ticker takes on this anchor: whichever sign carries more of the firing moves
    pos = int(((vals > 0) & fired).sum())
    neg = int(((vals < 0) & fired).sum())
    side = 1 if pos >= neg else -1
    hits = fired & (np.sign(vals) == side)

    # where the bulk of the HITS actually lands - the operator's "5 5 5 6 5 6"
    hit_days = [int(pd.Timestamp(p[0], unit="D").day) for p, h in zip(picks, hits) if h]
    if hit_days:
        vals_u, cnts = np.unique(np.array(hit_days), return_counts=True)
        mode_day = int(vals_u[np.argmax(cnts)])
        mode_share = float(cnts.max()) / len(hit_days)
        day_spread = int(max(hit_days) - min(hit_days))
    else:
        mode_day, mode_share, day_spread = -1, 0.0, 99

    return {
        "n": n,
        "picks": tuple(int(p[1]) for p in picks),
        "hits": int(hits.sum()),
        "hit_rate": float(hits.sum()) / n,
        "side": "long" if side > 0 else "short",
        "mean_move": float(raw_o[hits].mean()) if hits.any() else 0.0,
        "mean_move_px": (float(raw2_o[hits].mean()) if (raw2_o is not None and hits.any()) else None),
        "mode_day": mode_day,
        "mode_share": mode_share,
        "day_spread": day_spread,
        "hit_days": hit_days,
        "recent_hit_rate": float(hits[-4:].mean()) if n >= 4 else float(hits.mean()),
        "dates": [int(p[0]) for p in picks],
        "values": [float(v) for v in vals],
    }


def _anchor_tensor(dates, day_tol, min_occurrences):
    """-> (IDX, VALID, SLOT_OK, days) or None.

    IDX[a, s, k] is a session index; the k axis is sorted by distance from anchor a's own day, so
    "the firing session nearest the anchor day" is the FIRST firing k. VALID marks real entries in
    the padding, SLOT_OK marks slots (quarter-months) that exist for that anchor.
    """
    ordv = dates.view("int64") // 86_400_000_000_000
    dom = dates.day.to_numpy()
    y, m = dates.year.to_numpy(), dates.month.to_numpy()
    months = sorted(set(zip(y.tolist(), m.tolist())))

    anchors = []
    for D in range(1, 32):
        for P in PHASES:
            slots = []
            for (yy, mm) in months:
                if mm % QUARTER_STEP_MONTHS != P:
                    continue
                try:
                    centre = pd.Timestamp(year=yy, month=mm, day=D)
                except ValueError:
                    continue
                c = centre.value // 86_400_000_000_000
                idx = np.flatnonzero(np.abs(ordv - c) <= day_tol)
                if idx.size:
                    idx = idx[np.argsort(np.abs(dom[idx] - D), kind="stable")]
                    slots.append(idx)
            if len(slots) >= min_occurrences:
                anchors.append((D, P, slots))
    if not anchors:
        return None

    A = len(anchors)
    S = max(len(sl) for _, _, sl in anchors)
    K = max((len(i) for _, _, sl in anchors for i in sl), default=1)
    IDX = np.zeros((A, S, K), dtype=np.int32)
    VALID = np.zeros((A, S, K), dtype=bool)
    SLOT_OK = np.zeros((A, S), dtype=bool)
    for a, (_D, _P, slots) in enumerate(anchors):
        for sidx, idx in enumerate(slots):
            SLOT_OK[a, sidx] = True
            IDX[a, sidx, :len(idx)] = idx
            VALID[a, sidx, :len(idx)] = True
    return IDX, VALID, SLOT_OK, np.array([a[0] for a in anchors])


def _best_rate(z, tensor, min_abs_z, z2=None, min_abs_z2=None):
    """The best hit rate any anchor reaches on `z` - the whole search, vectorised.

    With z2 given an occurrence must clear both floors and agree in sign, exactly as
    _score_anchor's own `_fires` requires."""
    IDX, VALID, SLOT_OK, _ = tensor
    vals = z[IDX]                                   # (A, S, K)
    fires = (np.abs(vals) >= min_abs_z) & VALID
    if z2 is not None:
        v2 = z2[IDX]
        fires &= (np.abs(v2) >= min_abs_z2) & (np.sign(v2) == np.sign(vals))
    first = np.argmax(fires, axis=2)                # 0 when none fire -> the nearest session
    sel = np.take_along_axis(vals, first[:, :, None], axis=2)[:, :, 0]
    sel_fire = np.take_along_axis(fires, first[:, :, None], axis=2)[:, :, 0] & SLOT_OK
    pos = ((sel > 0) & sel_fire).sum(axis=1)
    neg = ((sel < 0) & sel_fire).sum(axis=1)
    side = np.where(pos >= neg, 1.0, -1.0)[:, None]
    hits = (sel_fire & (np.sign(sel) == side)).sum(axis=1)
    n = SLOT_OK.sum(axis=1)
    with np.errstate(invalid="ignore", divide="ignore"):
        rate = np.where(n > 0, hits / np.maximum(n, 1), 0.0)
    return float(rate.max()) if rate.size else 0.0


# =================================================================================================
# 3) stage one: the candidates
# =================================================================================================
def _collapse_neighbours(rows, day_tol):
    """One row per EVENT, not one per day-of-month that happens to reach it.

    An event on the 16th is inside +/-day_tol of anchors 14..18, so a single real pattern otherwise
    appears five times. Within one (ticker, phase) any two anchors whose centres sit within
    2*day_tol of each other overlap by construction and cannot be separate quarterly events, so
    they form a cluster; the survivor is the strongest, and among equals the one whose
    day-of-month is closest to the median day the event actually lands on.
    """
    if not len(rows):
        return rows
    keep = []
    for (_tk, _p), g in rows.groupby(["ticker", "month_phase"], sort=False):
        g = g.sort_values("day_of_month", kind="stable")
        cluster, last = [], None
        for _, r in g.iterrows():
            if last is not None and r["day_of_month"] - last > 2 * day_tol:
                keep.append(_best_of(cluster)); cluster = []
            cluster.append(r); last = r["day_of_month"]
        if cluster:
            keep.append(_best_of(cluster))
    return pd.DataFrame(keep).drop(columns=["_centre"], errors="ignore").reset_index(drop=True)


def _best_of(cluster):
    return sorted(cluster, key=lambda r: (-r["hit_rate"], -r["n"], abs(r["day_of_month"] - r["_centre"])))[0]


def find_quarterly(bars, *, day_tol=2, min_occurrences=6, min_hit_rate=0.70,
                   min_abs_z=1.5, min_abs_z_px=1.0, require_both=True,
                   standardise=True, min_mode_share=0.60, max_day_spread=2,
                   progress_every=500):
    """-> DataFrame, one row per (ticker, anchor) that clears the filters.

    min_abs_z is what "the event happened" MEANS, and it is the setting to reach for first.
    With standardise=True (default) it is in robust sigmas of the ticker's own 09:30->09:45 move,
    so every ticker is gated on the same terms; with False it is raw percentage points of
    `deviation`, which is only comparable within one ticker.

    The default 1.5 is deliberately well above the noise: at 0.5 some 62% of ordinary sessions
    clear the bar, the floor stops marking anything, and the 70% rule degenerates into measuring
    sign agreement - on the self-test fixture that alone produced four pure-noise anchors at
    73-82%. Lower it only with the null test in hand.

    min_mode_share / max_day_spread are the "almost the same day" requirement. mode_share is the
    share of HITS landing on the single most common day-of-month, day_spread the gap between the
    earliest and latest. "5 5 5 6 5 6" scores mode_share 0.67 and spread 1 and passes the defaults;
    a 3/5/7/4/6 scatter scores 0.20 and does not.
    """
    rows = []
    gb = bars.groupby("ticker", sort=True)
    for i, (tk, g) in enumerate(gb):
        if progress_every and i and i % progress_every == 0:
            print("  scanned %d tickers, %d candidates so far" % (i, len(rows)))
        if len(g) < min_occurrences:
            continue
        g = g.sort_values("day", kind="stable")
        dates = pd.DatetimeIndex(pd.to_datetime(g["day"].to_numpy()))
        raw = g["move"].to_numpy(dtype="float64")
        raw2 = g["move_px"].to_numpy(dtype="float64") if ("move_px" in g.columns and require_both) else None
        day_of_month = dates.day.to_numpy()
        if standardise:
            sd = _robust_scale(raw)
            if sd <= 0:
                continue                      # a ticker that never moves has no pattern to find
            z = raw / sd
        else:
            z = raw
        z2 = None
        if raw2 is not None:
            if standardise:
                sd2 = _robust_scale(raw2)
                if sd2 <= 0:
                    continue
                z2 = raw2 / sd2
            else:
                z2 = raw2
        slots = _anchor_slots(dates, day_tol)
        for (D, P), per_month in slots.items():
            if len(per_month) < min_occurrences:
                continue
            s = _score_anchor(z, per_month, min_abs_z, raw=raw, anchor_day=D, days=day_of_month,
                              z2=z2, raw2=raw2, min_abs_z2=min_abs_z_px)
            if s is None or s["n"] < min_occurrences or s["hit_rate"] < min_hit_rate:
                continue
            if s["mode_share"] < min_mode_share or s["day_spread"] > max_day_spread:
                continue
            last_day = pd.Timestamp(s["dates"][-1], unit="D")
            rows.append({
                "ticker": tk, "day_of_month": D, "month_phase": P,
                "months": ",".join(str(mm) for mm in range(1, 13) if mm % QUARTER_STEP_MONTHS == P),
                "n": s["n"], "hits": s["hits"], "hit_rate": round(s["hit_rate"], 4),
                "side": s["side"], "mean_move": round(s["mean_move"], 4),
                "mean_move_px": (round(s["mean_move_px"], 4) if s["mean_move_px"] is not None else None),
                "mode_day": s["mode_day"], "mode_share": round(s["mode_share"], 3),
                "day_spread": s["day_spread"], "hit_days": s["hit_days"],
                "recent_hit_rate": round(s["recent_hit_rate"], 4),
                "last_date": last_day.strftime("%Y-%m-%d"),
                "next_expected": (last_day + pd.DateOffset(months=QUARTER_STEP_MONTHS)).strftime("%Y-%m-%d"),
                "values": [round(v, 3) for v in s["values"]],
                "_centre": float(np.median([pd.Timestamp(c, unit="D").day for c in s["dates"]])),
            })
    out = _collapse_neighbours(pd.DataFrame(rows), day_tol)
    if len(out):
        # The majority of the WHOLE sample is the criterion, so hit_rate leads; n breaks ties,
        # because 8 of 10 is a stronger statement than 5 of 6 at the same rate.
        out = out.sort_values(["hit_rate", "n", "mode_share"],
                              ascending=[False, False, False]).reset_index(drop=True)
    return out


# =================================================================================================
# 4) stage two: is it better than the same ticker shuffled?
# =================================================================================================
def quarterly_pvalues(bars, candidates, *, day_tol=2, min_occurrences=6, min_abs_z=1.5,
                      min_abs_z_px=1.0, require_both=True,
                      standardise=True, n_null=400, min_null=None, seed=0, progress_every=200):
    """-> `candidates` with p_value and n_null added.

    The null keeps the ticker's own dates and its own distribution of moves and only breaks the
    link between them: the moves are permuted across its sessions and the WHOLE anchor search is
    re-run. p is the share of shuffles whose BEST anchor scores at least as well as the observed
    one, so it prices in the fact that the real search also got to pick its best of ~93.

    Run once per TICKER, not once per candidate: the null distribution depends on the ticker's own
    moves and dates, never on which anchor won.

    min_null switches on early stopping: a candidate already matched often enough to be out of
    reach of p <= 0.05 stops drawing. It is OFF by default because the vectorised search made the
    whole null cost about a minute - and near p = 0.05 an early stop flips a verdict now and then,
    which is not worth trading for seconds. Set it (60 is sensible) only if the run grows.

    A p of 0.05 over hundreds of tickers still leaves false positives - read it together with n
    and hit_rate, not alone.
    """
    if not len(candidates):
        out = candidates.copy()
        out["p_value"] = []
        out["n_null"] = []
        return out

    rng = np.random.default_rng(seed)
    by_ticker = {}
    for i, (_, row) in enumerate(candidates.iterrows()):
        by_ticker.setdefault(row["ticker"], []).append((i, float(row["hit_rate"])))

    pvals = [float("nan")] * len(candidates)
    nulls = [0] * len(candidates)
    for t, (tk, wanted) in enumerate(by_ticker.items()):
        if progress_every and t and t % progress_every == 0:
            print("  null-tested %d/%d tickers" % (t, len(by_ticker)))
        g = bars[bars["ticker"] == tk].sort_values("day", kind="stable")
        dates = pd.DatetimeIndex(pd.to_datetime(g["day"].to_numpy()))
        z = g["move"].to_numpy(dtype="float64")
        z2 = g["move_px"].to_numpy(dtype="float64") if ("move_px" in g.columns and require_both) else None
        if standardise:
            sd = _robust_scale(z)
            if sd <= 0:
                continue
            z = z / sd
            if z2 is not None:
                sd2 = _robust_scale(z2)
                if sd2 <= 0:
                    continue
                z2 = z2 / sd2
        tensor = _anchor_tensor(dates, day_tol, min_occurrences)
        if tensor is None:
            continue

        obs = np.array([w[1] for w in wanted], dtype="float64")
        ge = np.zeros(len(wanted), dtype=np.int64)
        done = np.zeros(len(wanted), dtype=bool)
        order = np.arange(len(z))
        drawn = 0
        for d in range(n_null):
            # ONE permutation applied to both axes: a session's DEV and its Stack% belong together,
            # and shuffling them apart would test an easier null than the search actually faces.
            rng.shuffle(order)
            zz = z[order]
            zz2 = z2[order] if z2 is not None else None
            best = _best_rate(zz, tensor, min_abs_z, z2=zz2, min_abs_z2=min_abs_z_px)
            ge += (best >= obs) & ~done
            drawn = d + 1
            if min_null and drawn >= min_null:
                # a candidate already matched this often cannot reach p <= 0.05
                done |= (ge + 1) / (drawn + 1) > 0.05
                if done.all():
                    break
        for (i, _r), g_, in zip(wanted, ge):
            pvals[i] = (g_ + 1) / (drawn + 1)
            nulls[i] = drawn

    out = candidates.copy()
    out["p_value"] = [round(p, 4) if p == p else None for p in pvals]
    out["n_null"] = nulls
    return out.sort_values(["p_value", "hit_rate", "n"],
                           ascending=[True, False, False]).reset_index(drop=True)


In [ ]:
# -*- coding: utf-8 -*-
# -- self-test on hand-built sessions (runs WITHOUT final.parquet) -------------------------------
# Every check below is a property of the SEARCH, not of any ticker: a planted quarterly pattern
# must be found once (not once per neighbouring day-of-month), read in the right direction, and
# survive its own shuffle - while a half-flipped plant and pure noise must not.

rng = np.random.default_rng(7)
ok = []


def sessions(start="2024-02-21", end="2026-11-12"):
    d = pd.bdate_range(start, end)
    return [x.strftime("%Y-%m-%d") for x in d]


DAYS = sessions()


def build(ticker, event_dates, size, noise=0.25, px_scale=1.0):
    """Noise everywhere, a planted move on the event dates - on BOTH axes.

    move    = the deviation axis (the ticker's own move)
    move_px = the Stack% axis; px_scale 0 makes the price axis pure noise, which is how the tests
              check that an event invisible in price is rejected."""
    mv = rng.normal(0, noise, len(DAYS))
    px = rng.normal(0, noise, len(DAYS))
    for ed in event_dates:
        if ed in DAYS:
            i = DAYS.index(ed)
            mv[i] = size
            px[i] = size * px_scale
    return pd.DataFrame({"ticker": ticker, "day": DAYS, "move": mv, "move_px": px})


def check(name, cond, extra=""):
    print(("  ok   " if cond else "  FAIL ") + name + (("  " + str(extra)) if extra else ""))
    ok.append(cond)


# ---- 1. a clean quarterly long on day 16, phase 2 (the operator's own shape) -------------------
ev = []
for y in (2024, 2025, 2026):
    for m in (2, 5, 8, 11):
        for off in (0, -1, 1, -2, 2):                       # nearest business day to the 16th
            c = pd.Timestamp(year=y, month=m, day=16) + pd.Timedelta(days=off)
            s = c.strftime("%Y-%m-%d")
            if s in DAYS:
                ev.append(s); break
planted = build("AAA", ev, size=+2.0)
res = find_quarterly(planted, min_occurrences=6, min_hit_rate=0.70, min_abs_z=1.5, progress_every=0)
top = res.iloc[0] if len(res) else None
check("the planted anchor is found", top is not None and abs(top["day_of_month"] - 16) <= 2 and top["month_phase"] == 2,
      "" if top is None else "day=%d phase=%d n=%d rate=%.2f side=%s" % (top["day_of_month"], top["month_phase"], top["n"], top["hit_rate"], top["side"]))
check("one row per event, not one per neighbouring day", len(res) == 1, "rows=%d" % len(res))
check("direction is read as long", top is not None and top["side"] == "long")
check("hit rate is 1.00 on a clean plant", top is not None and top["hit_rate"] == 1.0)
check("the whole sample is one-sided", top is not None and top["hits"] == top["n"],
      "" if top is None else "%d/%d" % (top["hits"], top["n"]))
check("next_expected is 3 months after the last", top is not None and
      pd.Timestamp(top["next_expected"]) == pd.Timestamp(top["last_date"]) + pd.DateOffset(months=3),
      "" if top is None else "%s -> %s" % (top["last_date"], top["next_expected"]))

# ---- 2. a SHORT plant is read as short ---------------------------------------------------------
res2 = find_quarterly(build("BBB", ev, size=-2.0), min_occurrences=6, min_hit_rate=0.70, min_abs_z=1.5, progress_every=0)
check("a negative plant is read as short", len(res2) and res2.iloc[0]["side"] == "short")

# ---- 3. mixed direction must NOT reach 70% -----------------------------------------------------
mixed = build("CCC", ev, size=+2.0)
flip = [i for i, d in enumerate(DAYS) if d in ev][:5]        # flip ~half of them negative
mixed.loc[flip, "move"] = -2.0
res3 = find_quarterly(mixed, min_occurrences=6, min_hit_rate=0.70, min_abs_z=1.5, progress_every=0)
hit = res3[(res3["month_phase"] == 2) & ((res3["day_of_month"] - 16).abs() <= 2)] if len(res3) else res3
check("a half-flipped plant is rejected at 70%", len(hit) == 0, "rows for that anchor: %d" % len(hit))

# ---- 4. the +/-2 day tolerance actually catches a weekend shift --------------------------------
shifted = []
for y in (2024, 2025, 2026):
    for m in (2, 5, 8, 11):
        c = pd.Timestamp(year=y, month=m, day=16)
        for off in (2, -2, 1, -1, 0):                        # prefer the FAR edge of the window
            s = (c + pd.Timedelta(days=off)).strftime("%Y-%m-%d")
            if s in DAYS:
                shifted.append(s); break
res4 = find_quarterly(build("DDD", shifted, size=+2.0), min_occurrences=6, min_hit_rate=0.70, min_abs_z=1.5, progress_every=0)
hit4 = res4[(res4["month_phase"] == 2) & ((res4["day_of_month"] - 16).abs() <= 2)] if len(res4) else res4
check("events parked on the window edge are still matched", len(hit4) == 1 and hit4.iloc[0]["hit_rate"] >= 0.7,
      "" if not len(hit4) else "n=%d rate=%.2f" % (hit4.iloc[0]["n"], hit4.iloc[0]["hit_rate"]))

# ---- 5. PURE NOISE must not survive the null ---------------------------------------------------
noise_only = pd.DataFrame({"ticker": "EEE", "day": DAYS, "move": rng.normal(0, 1.0, len(DAYS)),
                           "move_px": rng.normal(0, 1.0, len(DAYS))})
cand = find_quarterly(noise_only, min_occurrences=6, min_hit_rate=0.70, min_abs_z=1.5, progress_every=0)
print("  note  pure noise produced %d raw candidates at 70%% - this is why stage two exists" % len(cand))
if len(cand):
    pv = quarterly_pvalues(noise_only, cand.head(5), n_null=200, min_abs_z=1.5, progress_every=0)
    worst = pv["p_value"].min()
    check("noise candidates do not get a small p-value", worst > 0.05, "best p among them: %.3f" % worst)
else:
    check("noise produced no candidates at all", True)

# ---- 6. the real plant DOES beat its own shuffle ------------------------------------------------
pv6 = quarterly_pvalues(planted, res.head(1), n_null=200, min_abs_z=1.5, progress_every=0)
check("the planted pattern beats its own shuffle", pv6.iloc[0]["p_value"] <= 0.01,
      "p=%.4f" % pv6.iloc[0]["p_value"])


# ---- 7/8/9: "almost the same day" ---------------------------------------------------------------
def on_days(day_seq, phase_months=(1, 4, 7, 10)):
    """One event per quarter on the given day-of-month; nearest session when it is a weekend."""
    out, k = [], 0
    for y in (2024, 2025, 2026):
        for m in phase_months:
            if k >= len(day_seq):
                break
            try:
                c = pd.Timestamp(year=y, month=m, day=day_seq[k])
            except ValueError:
                k += 1
                continue
            for off in (0, 1, -1, 2, -2):
                sd = (c + pd.Timedelta(days=off)).strftime("%Y-%m-%d")
                if sd in DAYS:
                    out.append(sd)
                    break
            k += 1
    return out


tight = on_days([5, 5, 5, 6, 5, 6, 5, 5, 6, 5, 5])
res7 = find_quarterly(build("FFF", tight, size=+2.0), min_occurrences=6, min_hit_rate=0.70,
                      min_abs_z=1.5, progress_every=0)
check("5 5 5 6 5 6 passes the concentration filter", len(res7) >= 1,
      "" if not len(res7) else "mode_day=%d share=%.2f spread=%d days=%s"
      % (res7.iloc[0]["mode_day"], res7.iloc[0]["mode_share"], res7.iloc[0]["day_spread"], res7.iloc[0]["hit_days"]))

scatter = on_days([3, 7, 4, 8, 2, 9, 5, 1, 8, 3, 7])
res8 = find_quarterly(build("GGG", scatter, size=+2.0), min_occurrences=6, min_hit_rate=0.70,
                      min_abs_z=1.5, progress_every=0)
check("a scattered day-of-month is rejected", len(res8) == 0,
      "" if not len(res8) else "leaked: share=%.2f spread=%d days=%s"
      % (res8.iloc[0]["mode_share"], res8.iloc[0]["day_spread"], res8.iloc[0]["hit_days"]))

# A real +2.0 event on the 5th every quarter, plus a LOUDER +3.0 three days away. The occurrence
# must still be read on the 5th: the old largest-|move| rule took the neighbour and scattered the
# day series of a perfectly regular event.
base = build("HHH", on_days([5] * 11), size=+2.0)
for _d in on_days([8] * 11):
    base.loc[base["day"] == _d, "move"] = +3.0
res9 = find_quarterly(base, min_occurrences=6, min_hit_rate=0.70, min_abs_z=1.5, progress_every=0)
check("a louder neighbour does not steal the day",
      len(res9) >= 1 and 4 <= res9.iloc[0]["mode_day"] <= 6,
      "" if not len(res9) else "mode_day=%d days=%s" % (res9.iloc[0]["mode_day"], res9.iloc[0]["hit_days"]))


# ---- 10. the second axis actually gates ---------------------------------------------------------
# Same planted quarterly pattern, but the PRICE axis is pure noise - the stock did not really move,
# the benchmark did. require_both must reject it, and switching the gate off must bring it back.
devonly = build("III", ev, size=+2.0, px_scale=0.0)
r10a = find_quarterly(devonly, min_occurrences=6, min_hit_rate=0.70, min_abs_z=1.5, progress_every=0)
r10b = find_quarterly(devonly, min_occurrences=6, min_hit_rate=0.70, min_abs_z=1.5,
                      require_both=False, progress_every=0)
check("an event invisible in Stack% is rejected", len(r10a) == 0, "rows=%d" % len(r10a))
check("...and reappears when the price gate is off", len(r10b) >= 1, "rows=%d" % len(r10b))


# ---- 11. the MAJORITY of the whole sample is what decides ---------------------------------------
# Seven of ten one way, three the other: 0.70 exactly, so it passes at the default bar and fails
# the moment the bar is raised. Nothing about WHERE in the series the minority sits may matter -
# the three odd ones are deliberately the most recent, which the old last-3 rule would have killed.
maj = build("JJJ", ev, size=+2.0)
_last3 = [d for d in ev if d in DAYS][-3:]
for _d in _last3:
    maj.loc[maj["day"] == _d, ["move", "move_px"]] = -2.0
r11 = find_quarterly(maj, min_occurrences=6, min_hit_rate=0.70, min_abs_z=1.5, progress_every=0)
got11 = r11[(r11["month_phase"] == 2) & ((r11["mode_day"] - 16).abs() <= 2)] if len(r11) else r11
check("a 7-of-10 majority passes even when the last 3 disagree", len(got11) >= 1,
      "" if not len(got11) else "%d/%d = %.2f" % (got11.iloc[0]["hits"], got11.iloc[0]["n"], got11.iloc[0]["hit_rate"]))
r11b = find_quarterly(maj, min_occurrences=6, min_hit_rate=0.85, min_abs_z=1.5, progress_every=0)
got11b = r11b[(r11b["month_phase"] == 2) & ((r11b["mode_day"] - 16).abs() <= 2)] if len(r11b) else r11b
check("...and fails once the majority bar is raised to 85%", len(got11b) == 0, "rows=%d" % len(got11b))

print("\n%s  (%d/%d)" % ("ALL CHECKS PASSED" if all(ok) else "*** FAILURES ***", sum(ok), len(ok)))


In [ ]:
# -- run on final.parquet -------------------------------------------------------------------------
# PATHS: identical to every other notebook here - FINAL_PARQUET_PATH / SIGNALS_DIR win, else
# ORION_HOME, else the OriON folder found by walking up from the cwd.
from pathlib import Path
import os


def _resolve_orion_paths(strategy_code: str):
    final_env = os.environ.get("FINAL_PARQUET_PATH")
    sig_env = os.environ.get("SIGNALS_DIR")
    orion_env = os.environ.get("ORION_HOME")

    final_path = Path(final_env).expanduser().resolve() if final_env else None
    signals_base = Path(sig_env).expanduser().resolve() if sig_env else None

    if (final_path is None or signals_base is None) and orion_env:
        orion_home = Path(orion_env).expanduser().resolve()
        if final_path is None:
            final_path = (orion_home / "CRACEN" / "final.parquet").resolve()
        if signals_base is None:
            signals_base = (orion_home / "signals").resolve()

    if final_path is None or signals_base is None:
        here = Path.cwd().resolve()
        orion_home = None
        for parent in [here] + list(here.parents):
            if parent.name.lower() == "orion":
                orion_home = parent
                break
            cand = parent / "OriON"
            if cand.exists() and cand.is_dir():
                orion_home = cand.resolve()
                break
        if orion_home is None:
            raise RuntimeError("Cannot locate OriON. Set ORION_HOME env var (recommended).")
        if final_path is None:
            final_path = (orion_home / "CRACEN" / "final.parquet").resolve()
        if signals_base is None:
            signals_base = (orion_home / "signals").resolve()

    out_dir = (signals_base / strategy_code.lower()).resolve()
    out_dir.mkdir(parents=True, exist_ok=True)
    if not final_path.exists():
        raise FileNotFoundError(f"FINAL parquet not found: {final_path}")
    return final_path, out_dir


FINAL_PATH, OUT_DIR = _resolve_orion_paths("quarterly")
# ENTRY 09:30, EXIT 09:45 (the operator, 2026-11-17). The window is in the FILE NAME on purpose:
# retiming the search must not silently reuse bars measured over a different window, and a stale
# cache is invisible otherwise.
OPEN_MIN, LATE_MIN = 9 * 60 + 30, 9 * 60 + 45
BARS_CACHE = OUT_DIR / ("quarterly_bars_%04d_%04d.parquet"
                        % (OPEN_MIN // 60 * 100 + OPEN_MIN % 60, LATE_MIN // 60 * 100 + LATE_MIN % 60))

# ---- step 1: the 09:30 -> 09:45 move on both axes, one row per ticker-session ---------------
# Cached, because this is the only part that reads the whole file.
bars = None
if BARS_CACHE.exists():
    bars = pd.read_parquet(BARS_CACHE)
    if "move_px" not in bars.columns:
        print("cache has no move_px (single-axis) - rebuilding")
        bars = None
    else:
        print("reusing", BARS_CACHE, "-", f"{len(bars):,}", "ticker-sessions")
if bars is None:
    bars = load_quarterly_bars(str(FINAL_PATH), dev_field="deviation", px_field="Stack%",
                               open_min=OPEN_MIN, late_min=LATE_MIN)
    bars.to_parquet(BARS_CACHE, index=False)
    print("built", BARS_CACHE, "-", f"{len(bars):,}", "ticker-sessions")

print("tickers : %s" % f"{bars['ticker'].nunique():,}")
print("sessions: %s   (%s .. %s)" % (f"{bars['day'].nunique():,}", bars["day"].min(), bars["day"].max()))
print("DEV move    : sigma %.3f | mean|move| %.3f" % (bars["move"].std(), bars["move"].abs().mean()))
print("Stack%% move : sigma %.3f | mean|move| %.3f" % (bars["move_px"].std(), bars["move_px"].abs().mean()))

# Tickers with big holes cannot show a full set of quarters whatever they do, and their n would
# understate a pattern rather than overstate it. 400 of the ~686 sessions is a loose floor.
_cnt = bars.groupby("ticker")["day"].count()
_keep = set(_cnt[_cnt >= 400].index)
print("tickers with >= 400 sessions: %s  (dropping %s thin ones)"
      % (f"{len(_keep):,}", f"{bars['ticker'].nunique() - len(_keep):,}"))
bars = bars[bars["ticker"].isin(_keep)]

# ---- step 2: candidates ----------------------------------------------------------------------
# min_abs_z is what "the event happened" MEANS. deviation is in percentage points, which are not
# comparable between tickers, so the search divides each ticker's moves by its own ROBUST sigma
# (1.4826*MAD - not inflated by the very event days being hunted) and gates in those units.
# 1.5 is well clear of the noise; at 0.5 some 62% of ordinary sessions clear the bar and the 70%
# rule stops measuring anything but sign agreement.
cand = find_quarterly(
    bars,
    day_tol=2,              # the operator's own +/-2 calendar days
    min_occurrences=6,      # at least 6 of the ~11 quarters in this file produced a session
    # BOTH axes must fire, in the same direction. deviation says the move is the ticker's own,
    # Stack% says the price actually went somewhere; either alone misleads (deviation fires when
    # the stock holds still and the benchmark drops, Stack% fires on every market-wide morning).
    # Each is standardised by its OWN robust sigma, so the two floors mean the same thing.
    min_abs_z=1.5,        # the DEV floor
    min_abs_z_px=1.0,     # the Stack% floor
    require_both=True,    # False searches on deviation alone
    # THE DIRECTION TEST: the share of ALL occurrences that went one way. 0.70 of ten is seven,
    # and it does not matter where in the series the other three sit.
    min_hit_rate=0.70,
    # "almost the same day": mode_share is the share of HITS on the single most common
    # day-of-month, day_spread the gap between earliest and latest. "5 5 5 6 5 6" is
    # share 0.67 / spread 1 and passes; a 3/5/7/4/6 scatter is share 0.20 and does not.
    min_mode_share=0.60,
    max_day_spread=2,
)
print("\ncandidates at 70%%: %s over %s tickers" % (f"{len(cand):,}", f"{cand['ticker'].nunique():,}" if len(cand) else 0))

# ---- step 3: is any of it better than the same ticker shuffled? ------------------------------
# READ THIS BEFORE TRADING ANY OF IT. On pure noise one ticker produces several 70% anchors, so
# the raw list above is mostly luck. The null re-runs the whole 93-anchor search on this ticker's
# own moves, permuted across its own dates.
ranked = quarterly_pvalues(bars, cand, n_null=400, min_abs_z=1.5, min_abs_z_px=1.0)
survivors = ranked[ranked["p_value"] <= 0.05] if len(ranked) else ranked
print("survive p <= 0.05: %s" % (f"{len(survivors):,}" if len(ranked) else 0))

OUT_CSV = OUT_DIR / "quarterly_candidates.csv"
ranked.to_csv(OUT_CSV, index=False)
print("written ->", OUT_CSV)

# ---- what the operator asked to see first: the freshest, most one-sided ones ------------------
if len(survivors):
    fresh = survivors.sort_values(["hit_rate", "n"], ascending=[False, False])
    print("\n=== strongest majorities first ===")
    cols = ["ticker", "mode_day", "months", "side", "n", "hits", "hit_rate",
            "mean_move", "mean_move_px", "day_spread", "last_date", "next_expected", "p_value"]
    # mean_move is the DEV move in pp, mean_move_px the Stack% move in pp
    print(fresh[cols].head(40).to_string(index=False))
